####Working with timestamp
1. [Timestamp functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)

####1. How Spark stores the timestamp

1. Timestamp is internally stored as 12 byte integer known as INT96
2. Timestamp is made up of 7 fields
    1. Year
    2. Month
    3. Day
    4. Hour
    5. Minute
    6. Second
        * Up to 6 decimal places
        * Microsecond precision
    7. Timezone

####2. Requirement
You are given the below dataframes

In [0]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")]
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))
display(df_3)

2.1 covert the df_1 to timestamp

In [0]:
from pyspark.sql.functions import to_timestamp, col

df_1.select("string_time",
             to_timestamp("string_time", "yyyy-MM-dd'T'HH:mm:ss.SSSS").alias("valid_time")
             ).display()

2.2 Convert the df_2 to time

In [0]:
df_2.selectExpr(
    "string_time",
    "to_timestamp(string_time, 'dd-MM-yyyy HH:mm:ss.SSSS') as valid_time"
).display()

2.3 Convert the df_3 to time


In [0]:
df_3.selectExpr(
    "string_time",
    "try_to_timestamp(string_time, 'dd-MM-yyyy HH:mm:ss.SSSS') as valid_time"
).display()


####3. Timezone information

1. A timestamp without timezone information is incomplete.
2. Spark offers two data types for timestamp
    1. TIMESTAMP
    2. TIMESTAMP_NTZ
3. For TIMESTAMP, Spark assumes session timezone as the default when timezone is not specified
4. Session timezone is specified as spark.sql.session.timeZone

3.1 What is your default session timezone?


In [0]:
spark.conf.get("spark.sql.session.timeZone")

3.2 Change your session timezone to IST

In [0]:
spark.conf.set("spark.sql.session.timeZone", 'Etc/UTC')

####4. Working with NTZ data

4.1 Load machine-events-no-tz.csv file and show the data

In [0]:
event_ntz_schema = "component string, event_time string, reading string"

event_ntz_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(event_ntz_schema)
        .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/machine-events-no-tz.csv")
)

event_ntz_df.display()

4.2 Parse te event_time to a TIMESTAMP_NTZ value

In [0]:
from pyspark.sql.functions import to_timestamp_ntz, lit

event_valid_ntz_df = (
    event_ntz_df.withColumn("event_time_valid_ntz", to_timestamp_ntz("event_time", lit("dd-MM-yyyy HH:mm:ss.SSS")))
)

event_valid_ntz_df.display()

4.3 event_time_ntz field to a valid timestamp value\
Assume the event_time_ntz is IST time

In [0]:
from pyspark.sql.functions import convert_timezone, lit

stz = spark.conf.get("spark.sql.session.timeZone")

events_df = (
    event_valid_ntz_df.withColumn("event_time_tz", to_timestamp(convert_timezone(lit("IST"), lit(stz), "event_time_valid_ntz")))
)

events_df.display()

####5. Working with TZ data

5.1 Load machine-events-with-tz.csv file and show the data

In [0]:
event_tz_schema = "component string, event_time_tz_str string, reading string"

events_tz_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(event_tz_schema)
    .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/machine-events-with-tz.csv")
)

display(events_tz_df)

5.2 Parse the event_time field to a valid timestamp value\
Timezone information is provided in the data file

In [0]:
from pyspark.sql.functions import to_timestamp
event_data_df = (
    events_tz_df.withColumn("event_time_tz", to_timestamp("event_time_tz_str", "dd-MM-yyyy HH:mm:ss.SSSZ"))
)
event_data_df.display()